# E30 公开调查筛选、缺失及指定模型有限复算

## tl;dr
按更正附件代码的标签筛选得到1534人；五档频数61/182/762/408/121，与原表相符。完整模型字段的完整行1531，三条伴侣教育“不知道”转为未映射缺失。另以Python比例优势模型复算11个指定模型、全模型23个斜率和固定完整行上的后向AIC路径；同一资料的跨实现核对不构成独立实验，也不推论因果或个人体验。

## Context & Methods
数据是CRAN TH.data公开源仓库的`inst/rda/CHFLS.rda`，固定提交3bf0832f84f436fe4c57f6cc0de08e298c8470b8；处理依据[更正附件R代码](https://cran.r-project.org/web/packages/multcomp/vignettes/chfls1.R)。原调查1999—2000年，下载2026-11-04；不把软件包当前版本当调查时间。

### Key Assumptions
先核筛选、频数与模型字段缺失，再核指定模型。代码`ZW04 != yes`允许七条legit skip与一条dont know通过，不能将其改称所有人均明确报告异性恋。原R代码对三种无关字段的重复标签警告忽略；本记录在转换因子时保留值标签、去重类别表，所用字段标签无此重复问题。模型按原R脚本先在1534行上除以样本标准差（ddof=1），再按模型删缺失。Python内部再中心化设计列以改善条件；这只平移阈值，不改变斜率或似然，因此不拿中心化阈值与R直接比较。类别首级为参照，无额外常数列。未运行原R脚本或max-t同时推断。

## Data
运行需要Python、numpy、pandas、rdata、xarray、scipy、statsmodels、patsy；`pip install numpy pandas rdata xarray scipy statsmodels patsy`。默认将公开数据下载到inputs/CHFLS.rda并验证SHA256；可用CHFLS_RDA指定已有副本、CHFLS_PYTHON_DEPS指定依赖目录。数据文件不随本记录发布，代码只打印汇总，不显示个体行。


### 1. 设置依赖


In [1]:
import sys, json
from pathlib import Path
import os, hashlib, urllib.request
if os.environ.get('CHFLS_PYTHON_DEPS'):
    sys.path.insert(0, os.environ['CHFLS_PYTHON_DEPS'])
import numpy as np
import pandas as pd
import rdata



### 2. 校验文件并转换标签


In [2]:
constructors = dict(rdata.conversion.DEFAULT_CLASS_MAP)
def factor_values(obj, attrs, ordered=False):
    levels = list(attrs['levels'])
    values = [None if np.ma.is_masked(i) or i < 1 else levels[int(i)-1] for i in obj]
    return pd.Categorical(values, categories=list(dict.fromkeys(levels)), ordered=ordered)
constructors['factor'] = factor_values
constructors['ordered'] = lambda obj, attrs: factor_values(obj, attrs, True)
data_path = Path(os.environ.get('CHFLS_RDA', 'inputs/CHFLS.rda'))
source_url = 'https://raw.githubusercontent.com/cran/TH.data/3bf0832f84f436fe4c57f6cc0de08e298c8470b8/inst/rda/CHFLS.rda'
if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(urllib.request.urlopen(source_url, timeout=30).read())
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == '38e0b90261ba214abd4edb7654ae7224c51842f0b2ce85150b4e66b68bda8dcc'
raw = rdata.read_rda(data_path, constructor_dict=constructors)['chfls1']


### 3. 按附件建立筛选与字段


In [3]:
orgasm_levels = ['never','rarely','sometimes','often','always']
keep = (raw.ZJ05 == 'female') & (raw.ZW04 != 'yes') & raw.AX16.isin(orgasm_levels)
cohort = raw.loc[keep].copy()
education_levels = ['univ/grad','j col','up mid','low mid','primary','no school']
def rank(series, levels):
    return series.astype(object).map({v: i+1 for i,v in enumerate(levels)}).astype(float)
frame = pd.DataFrame(index=cohort.index)
frame['orgasm'] = rank(cohort.AX16, orgasm_levels)
frame['AincomeSD'] = pd.to_numeric(cohort.INCAM, errors='coerce')
frame['AheightSD'] = pd.to_numeric(cohort.A14, errors='coerce').where(cohort.A14 >= 0)
frame['RAdurationSD'] = pd.to_numeric(cohort.ZJ06, errors='coerce') - pd.to_numeric(cohort.A35, errors='coerce')
frame['RageSD'] = pd.to_numeric(cohort.ZJ06, errors='coerce')
frame['Redu'] = rank(cohort.ZJ07, education_levels)
frame['edudiffSD'] = rank(cohort.A21, education_levels) - frame.Redu
frame['wealthdiffSD'] = pd.to_numeric(cohort.INCRM, errors='coerce') - pd.to_numeric(cohort.INCAM, errors='coerce')
frame['Rhealth'] = rank(cohort.JK01, ['poor','not good','fair','good','excellent'])
frame['Rhappy'] = rank(cohort.JK20, ['v unhappy','not too','relatively','very'])
frame['Region'] = rank(cohort.REGION6, ['CentralW','Northeast','North','InlandS','CoastalE','CoastalS'])


## Results
### 4. 频数及缺失汇总


In [4]:
counts = {v: int((cohort.AX16 == v).sum()) for v in orgasm_levels}
out = {
 'raw_shape': list(raw.shape), 'filtered_n': len(cohort), 'frequencies': counts,
 'high_two_n': counts['often']+counts['always'],
 'high_two_percent': 100*(counts['often']+counts['always'])/len(cohort),
 'missing_by_full_model_field': {c:int(n) for c,n in frame.isna().sum().items()},
 'full_model_complete_n': int(frame.notna().all(axis=1).sum()),
 'partner_education_unmapped': {str(k):int(v) for k,v in cohort.loc[frame.edudiffSD.isna(),'A21'].value_counts().items() if v},
 'initial_model_complete_n': int(frame[['orgasm','AincomeSD','AheightSD']].notna().all(axis=1).sum()),
 'independent_frequency_sum': sum(counts.values()),
 'filter_label_counts': {str(k):int(v) for k,v in cohort.ZW04.value_counts().items() if v},
}
assert sum(counts.values()) == len(cohort)
print(json.dumps(out,indent=2))


{
  "raw_shape": [
    3821,
    423
  ],
  "filtered_n": 1534,
  "frequencies": {
    "never": 61,
    "rarely": 182,
    "sometimes": 762,
    "often": 408,
    "always": 121
  },
  "high_two_n": 529,
  "high_two_percent": 34.485006518904825,
  "missing_by_full_model_field": {
    "orgasm": 0,
    "AincomeSD": 0,
    "AheightSD": 0,
    "RAdurationSD": 0,
    "RageSD": 0,
    "Redu": 0,
    "edudiffSD": 3,
    "wealthdiffSD": 0,
    "Rhealth": 0,
    "Rhappy": 0,
    "Region": 0
  },
  "full_model_complete_n": 1531,
  "partner_education_unmapped": {
    "dont know": 3
  },
  "initial_model_complete_n": 1534,
  "independent_frequency_sum": 1534,
  "filter_label_counts": {
    "no": 1526,
    "legit skip": 7,
    "dont know": 1
  }
}


### 5. 指定比例优势模型与似然常数

实现依据[statsmodels官方OrderedModel示例](https://www.statsmodels.org/stable/examples/notebooks/generated/ordinal_regression.html)。BFGS最多2000次；保存收敛状态及总对数似然数值梯度，不计算Wald或max-t的p值。同时列实际样本量BIC与附件脚本固定n=1534的BIC。分组似然常数用整组和逐类别计数的log阶乘独立计算；只打印汇总，不打印参与者行。


In [5]:
import math
from statsmodels.miscmodels.ordinal_model import OrderedModel
import scipy, statsmodels

# The R supplement divides by sample SD before model-specific complete-case deletion.
numeric = ['AincomeSD','AheightSD','RAdurationSD','RageSD','edudiffSD','wealthdiffSD']
scaled = frame.copy()
for field in numeric:
    scaled[field] /= scaled[field].std(ddof=1)
factor_n = {'Redu':6,'Rhealth':5,'Rhappy':4,'Region':6}
specs = {
    'table1_start': ['AincomeSD','AheightSD'],
    'table1_step1': ['AincomeSD'],
    'table1_step2': ['AincomeSD','Rhappy'],
    'table2_step2': ['AincomeSD','Redu'],
    'table2_step3': ['AincomeSD','Redu','RageSD'],
    'table2_step4a': ['AincomeSD','Redu','RageSD','Rhappy'],
    'table2_step4b': ['AincomeSD','Redu','RageSD','edudiffSD'],
    'table2_step5': ['AincomeSD','Redu','RageSD','Rhappy','edudiffSD'],
    'table2_step6': ['AincomeSD','Redu','RageSD','Rhappy','edudiffSD','Region'],
    'table2_step7': ['AincomeSD','Redu','RageSD','Rhappy','edudiffSD','Region','Rhealth'],
    'full': numeric + ['Redu','Rhealth','Rhappy','Region'],
}
out = {'versions': {'numpy':np.__version__, 'pandas':pd.__version__,
                   'scipy':scipy.__version__, 'statsmodels':statsmodels.__version__},
       'scaling_sd_before_model_deletion': {f:float(frame[f].std(ddof=1)) for f in numeric},
       'models':{}}
for name, fields in specs.items():
    data = scaled[['orgasm'] + fields].dropna()
    x = pd.DataFrame(index=data.index)
    for field in fields:
        if field in factor_n:
            for level in range(2, factor_n[field]+1):
                x[f'{field}_{level}'] = (data[field] == level).astype(float)
        else:
            x[field] = data[field].astype(float)
    # Centering predictors only changes the four thresholds, improving conditioning.
    # Slopes, likelihood and AIC are invariant; do not compare centered thresholds to R.
    x = x - x.mean()
    model = OrderedModel(data.orgasm.astype(int)-1, x, distr='logit')
    fit = model.fit(method='bfgs', maxiter=2000, gtol=1e-7, disp=False, skip_hessian=True)
    k = len(fit.params)
    group_sizes = data.groupby(fields,observed=True,dropna=False).size()
    category_sizes = data.groupby(fields+['orgasm'],observed=True,dropna=False).size()
    log_constant = sum(math.lgamma(int(n)+1) for n in group_sizes) - sum(math.lgamma(int(n)+1) for n in category_sizes)
    item = {'n':len(data), 'k':k, 'minus_2_loglik':float(-2*fit.llf),
            'AIC':float(-2*fit.llf+2*k),
            'BIC_actual_n':float(-2*fit.llf+np.log(len(data))*k),
            'BIC_R_supplement_n1534':float(-2*fit.llf+np.log(1534)*k),
            'covariate_groups':len(group_sizes),
            'log_multinomial_constant':log_constant,
            'minus_2_grouped_loglik_with_constant':float(-2*fit.llf-2*log_constant),
            'converged':bool(fit.mle_retvals['converged']),
            'max_abs_score':float(np.max(np.abs(model.score(fit.params)))),
            'slopes':{f:float(fit.params[f]) for f in x.columns}}
    out['models'][name] = item
    print(name, json.dumps({k:v for k,v in item.items() if k != 'slopes'}), flush=True)



table1_start {"n": 1534, "k": 6, "minus_2_loglik": 3903.8039732433676, "AIC": 3915.8039732433676, "BIC_actual_n": 3947.817777134931, "BIC_R_supplement_n1534": 3947.817777134931, "covariate_groups": 332, "log_multinomial_constant": 1017.6964152869521, "minus_2_grouped_loglik_with_constant": 1868.4111426694635, "converged": true, "max_abs_score": 2.457657145778215e-05}


table1_step1 {"n": 1534, "k": 5, "minus_2_loglik": 3906.695292030963, "AIC": 3916.695292030963, "BIC_actual_n": 3943.373461940599, "BIC_R_supplement_n1534": 3943.373461940599, "covariate_groups": 28, "log_multinomial_constant": 1750.5721543440054, "minus_2_grouped_loglik_with_constant": 405.55098334295235, "converged": true, "max_abs_score": 9.577969561630022e-05}


table1_step2 {"n": 1534, "k": 8, "minus_2_loglik": 3880.5463177841634, "AIC": 3896.5463177841634, "BIC_actual_n": 3939.231389639581, "BIC_R_supplement_n1534": 3939.231389639581, "covariate_groups": 83, "log_multinomial_constant": 1564.0762252975974, "minus_2_grouped_loglik_with_constant": 752.3938671889687, "converged": true, "max_abs_score": 3.375680941959069e-05}


table2_step2 {"n": 1534, "k": 10, "minus_2_loglik": 3817.0303754784336, "AIC": 3837.0303754784336, "BIC_actual_n": 3890.3867152977055, "BIC_R_supplement_n1534": 3890.3867152977055, "covariate_groups": 110, "log_multinomial_constant": 1437.1343841413982, "minus_2_grouped_loglik_with_constant": 942.7616071956372, "converged": true, "max_abs_score": 4.9445483663257446e-05}


table2_step3 {"n": 1534, "k": 11, "minus_2_loglik": 3778.01698940836, "AIC": 3800.01698940836, "BIC_actual_n": 3858.7089632095594, "BIC_R_supplement_n1534": 3858.7089632095594, "covariate_groups": 966, "log_multinomial_constant": 318.2317570834505, "minus_2_grouped_loglik_with_constant": 3141.553475241459, "converged": true, "max_abs_score": 8.899392168210646e-05}


table2_step4a {"n": 1534, "k": 14, "minus_2_loglik": 3751.40582233828, "AIC": 3779.40582233828, "BIC_actual_n": 3854.1046980852607, "BIC_R_supplement_n1534": 3854.1046980852607, "covariate_groups": 1167, "log_multinomial_constant": 196.21424461687795, "minus_2_grouped_loglik_with_constant": 3358.977333104524, "converged": true, "max_abs_score": 8.813792501529432e-05}


table2_step4b {"n": 1531, "k": 12, "minus_2_loglik": 3760.6781334895422, "AIC": 3784.6781334895422, "BIC_actual_n": 3848.6822502374343, "BIC_R_supplement_n1534": 3848.7057412726685, "covariate_groups": 1242, "log_multinomial_constant": 141.45645756529728, "minus_2_grouped_loglik_with_constant": 3477.7652183589475, "converged": true, "max_abs_score": 4.2825462722467145e-05}


table2_step5 {"n": 1531, "k": 15, "minus_2_loglik": 3734.2820958341445, "AIC": 3764.2820958341445, "BIC_actual_n": 3844.2872417690096, "BIC_R_supplement_n1534": 3844.3166055630527, "covariate_groups": 1359, "log_multinomial_constant": 83.3873138882991, "minus_2_grouped_loglik_with_constant": 3567.5074680575462, "converged": true, "max_abs_score": 0.00011167419268966532}


table2_step6 {"n": 1531, "k": 20, "minus_2_loglik": 3719.2257744977046, "AIC": 3759.2257744977046, "BIC_actual_n": 3865.899302410858, "BIC_R_supplement_n1534": 3865.9384541362488, "covariate_groups": 1483, "log_multinomial_constant": 20.91219845245473, "minus_2_grouped_loglik_with_constant": 3677.401377592795, "converged": true, "max_abs_score": 0.0001528518186619563}


table2_step7 {"n": 1531, "k": 24, "minus_2_loglik": 3705.851968180609, "AIC": 3753.851968180609, "BIC_actual_n": 3881.8602016763934, "BIC_R_supplement_n1534": 3881.907183746862, "covariate_groups": 1505, "log_multinomial_constant": 11.09035488895912, "minus_2_grouped_loglik_with_constant": 3683.671258402691, "converged": true, "max_abs_score": 4.996428899993517e-05}


full {"n": 1531, "k": 27, "minus_2_loglik": 3705.2291563068393, "AIC": 3759.2291563068393, "BIC_actual_n": 3903.2384189895965, "BIC_R_supplement_n1534": 3903.291273818874, "covariate_groups": 1530, "log_multinomial_constant": 0.0, "minus_2_grouped_loglik_with_constant": 3705.2291563068393, "converged": true, "max_abs_score": 6.837783396347144e-05}


### 6. 固定完整行的后向AIC

每轮检验删除一个完整项，类别项整组删除；所有候选固定1531行。候选均需收敛，停止于没有单项删除可降低AIC。此为Python实现的后向删除，不冒称运行MASS::stepAIC，也没有重做所有前向候选搜索或样本外效度。


In [6]:

complete = scaled.dropna().copy()
def backward_candidate(fields):
    x = pd.DataFrame(index=complete.index)
    for field in fields:
        if field in factor_n:
            for level in range(2, factor_n[field]+1):
                x[f'{field}_{level}'] = (complete[field] == level).astype(float)
        else:
            x[field] = complete[field].astype(float)
    x -= x.mean()
    model = OrderedModel(complete.orgasm.astype(int)-1, x, distr='logit')
    fit = model.fit(method='bfgs', maxiter=2000, gtol=1e-7, disp=False, skip_hessian=True)
    return {'AIC':float(-2*fit.llf+2*len(fit.params)),
            'converged':bool(fit.mle_retvals['converged']),
            'max_abs_score':float(np.max(np.abs(model.score(fit.params))))}
remaining = list(specs['full'])
current = out['models']['full']['AIC']
path = [{'removed':None, 'AIC':current, 'remaining':remaining.copy()}]
all_candidates = []
while len(remaining)>1:
    candidates=[]
    for removed in remaining:
        result = backward_candidate([f for f in remaining if f != removed])
        assert result['converged'], (removed,result)
        candidates.append({'removed':removed, **result})
    all_candidates.append(candidates)
    best = min(candidates,key=lambda d:d['AIC'])
    if best['AIC'] >= current - 1e-7:
        break
    remaining.remove(best['removed'])
    current = best['AIC']
    path.append({**best,'remaining':remaining.copy()})
    print('backward',json.dumps(best),flush=True)
out['backward'] = {'n':len(complete),'path':path,'candidates':all_candidates,
                   'stopped_no_improving_single_term_removal':True}
print('backward complete',len(path),flush=True)


backward {"removed": "AheightSD", "AIC": 3757.2413986205775, "converged": true, "max_abs_score": 6.239033200325504e-05}


backward {"removed": "AincomeSD", "AIC": 3755.3015530387265, "converged": true, "max_abs_score": 1.6470307417586965e-05}


backward {"removed": "RAdurationSD", "AIC": 3753.766495306651, "converged": true, "max_abs_score": 8.217527504244665e-05}


backward {"removed": "wealthdiffSD", "AIC": 3752.7151605042673, "converged": true, "max_abs_score": 5.441482932533516e-05}


backward complete 5


### 7. 与附件显示值的有限核对

11个模型的AIC中，step4b的3784.7是本库计算值，原附件该位置只显示BIC，不能算该AIC已与原表逐项比对；其余10个有对应AIC。23个全模型斜率按表4两位小数核对。旧表分组似然起始值1868.1不能重现（本次1868.4），另外两行405.6/752.4相符，差异不擅自归因。


In [7]:
expected_aic = [3915.8,3916.7,3896.5,3837.0,3800.0,3779.4,3784.7,3764.3,3759.2,3753.9,3759.2]
actual_aic = [round(m['AIC'],1) for m in out['models'].values()]
assert actual_aic == expected_aic
assert all(m['converged'] for m in out['models'].values())
expected_slopes = [0.02,0.01,0.09,-0.37,-0.17,-0.03,0.11,0.14,-0.45,-0.98,-1.82,1.22,1.56,1.70,1.72,0.17,0.64,0.91,0.40,0.20,0.49,0.20,0.59]
assert [round(v,2) for v in out['models']['full']['slopes'].values()] == expected_slopes
assert [s['removed'] for s in path] == [None,'AheightSD','AincomeSD','RAdurationSD','wealthdiffSD']
assert [round(s['AIC'],1) for s in path] == [3759.2,3757.2,3755.3,3753.8,3752.7]
print(json.dumps({'specified_models_converged':len(out['models']), 'AIC_values':actual_aic,
 'full_slopes_rounded_match':len(expected_slopes),'backward_AIC':[round(s['AIC'],1) for s in path],
 'backward_candidate_fits':sum(map(len,all_candidates)),
 'grouped_table1_rounded':[round(out['models'][k]['minus_2_grouped_loglik_with_constant'],1) for k in ['table1_start','table1_step1','table1_step2']]},indent=2))


{
  "specified_models_converged": 11,
  "AIC_values": [
    3915.8,
    3916.7,
    3896.5,
    3837.0,
    3800.0,
    3779.4,
    3784.7,
    3764.3,
    3759.2,
    3753.9,
    3759.2
  ],
  "full_slopes_rounded_match": 23,
  "backward_AIC": [
    3759.2,
    3757.2,
    3755.3,
    3753.8,
    3752.7
  ],
  "backward_candidate_fits": 40,
  "grouped_table1_rounded": [
    1868.4,
    405.6,
    752.4
  ]
}


## Takeaways
本公开副本的频数与三条缺失已对应，指定模型收敛；10个已刊AIC显示值、全模型23斜率及后向删除路径相符。table2在加入教育差后由1534变1531，因此不能将跨不同观测行的AIC差值当纯粹变量贡献。后向比较固定1531行，收入在第二步删除；这支持更正附件所述分析结果，不证明收入在任何群体或情境中都无影响。

仍未完成：原R环境执行、所有前向候选搜索、max-t调整p值、比例优势假设诊断、插补敏感性及独立样本验证。旧表起始分组似然0.3差异保留。执行验证：七个代码单元以普通Python在同一新命名空间顺序运行，保存真实标准输出；JSON基本结构检查完成。未做Jupyter内核、正式nbformat验证或界面视觉检查。复核环境可安装Jupyter组件并运行`jupyter nbconvert --execute --to notebook --inplace E30_CHFLS筛选与缺失有限核对.ipynb`。数据与个人行不发布。


2026-11-04验证更新：本笔记已通过nbformat 5.11.1正式格式检查，并由nbclient 0.11.0在独立新Jupyter内核中从头执行，代码单元无错误输出。与先前普通Python执行的汇总标准输出逐字一致（允许消息分段不同）；已保存真实execution_count及聚合输出。此前仅普通Python执行的描述为阶段历史。尚未完成Notebook界面视觉验收；此次执行不扩大统计复现、因果或独立样本范围。